In [ ]:
import os
os.environ['GROQ_API_KEY'] = '...'

In [ ]:
!pip install -q langchain groq langchain-core requests

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests


In [ ]:
# tool create
@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b

In [ ]:
print(multiply.invoke({"a": 3, "b": 4}))

In [ ]:
multiply.name

In [ ]:
multiply.description


In [ ]:
multiply.args

In [ ]:
# tool binding : process of connecting your tool with your llm  
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

In [ ]:
llm_with_tools=llm.bind_tools([multiply])

In [ ]:
llm

In [ ]:
# tool calling
llm_with_tools.invoke('Hi how are you')

In [ ]:
query=HumanMessage(content='can you multiply 3 with 10')

In [ ]:
messages=[query]

In [ ]:
messages

In [ ]:
result=llm_with_tools.invoke(messages)

In [ ]:
result

In [ ]:
messages.append(result)

In [ ]:
messages

In [ ]:
# tool execution
tool_result=multiply.invoke(result.tool_calls[0])

In [ ]:
messages.append(tool_result)

In [ ]:
messages

In [ ]:
llm_with_tools.invoke(messages).content

In [ ]:
# Currency Conversion Tool
# create tool
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency:str,target_currency:str)->float:
    """ 
    This function fetches the currency conversion factor between a given base currency and target currency
    """
    url=f'https://v6.exchangerate-api.com/v6/e250bfe6899d6cb7e22a61de/pair/{base_currency}/{target_currency}'
    response=requests.get(url)
    return response.json()

@tool
def convert(base_currency_value:int,conversion_rate:Annotated[float, InjectedToolArg])->float:
    """ 
    Given a currency conversion rate this function calculatees the target currency value from a given base currency given value
    """
    return base_currency_value*conversion_rate

In [ ]:
get_conversion_factor.invoke({"base_currency":"USD","target_currency":"INR"})

In [ ]:
convert.invoke({"base_currency_value": 10, "conversion_rate": 95.77})

In [ ]:
# tool binding
llm=ChatGroq(model="llama-3.1-8b-instant")
llm_with_tools=llm.bind_tools([get_conversion_factor,convert])

In [ ]:
messages=[HumanMessage('What is the conversion factor between USD and INR,and based on that can you convert 10 USD to INR? ')]

In [ ]:
messages

In [ ]:
ai_messages=llm_with_tools.invoke(messages)

In [ ]:
ai_messages.tool_calls

In [ ]:
messages.append(ai_messages)

In [ ]:
import json
for tool_call in ai_messages.tool_calls:
    # execute the 1st tool and get the value of conversion rate 
    if tool_call['name']=='get_conversion_factor':
        tool_message1=get_conversion_factor.invoke(tool_call)
        print(tool_message1)
        # fetch this conversion rate
        conversion_rate=json.loads(tool_message1.content)['conversion_rate']
        # append this tool message to messages list
        messages.append(tool_message1)
    # execute the 2nd tool using the conversion rate from tool 1
    if tool_call['name']=='convert':
        # fetch the current arg
        tool_call['args']['conversion_rate']=conversion_rate
        tool_message2=convert.invoke(tool_call)
        messages.append(tool_message2)


In [ ]:
messages

In [ ]:
llm_with_tools.invoke(messages).content